# 05 — HOD-Mamba-Lite
**Reddit Opinion & Community Research — first working model **


In [1]:
import sys, importlib

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        get_ipython().system(f"pip install -q {pkg}")

for pkg, imp in [("torch", None), ("numpy", None), ("scikit-learn", "sklearn")]:
    ensure(pkg, imp)

print("Packages OK.")


Packages OK.


In [2]:
import os, json, pickle, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, classification_report

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = "/content/drive/MyDrive/PhD/reddit_opinion_research"
else:
    BASE_DIR = os.environ.get("REDDIT_RESEARCH_BASE_DIR", "./reddit_opinion_research")

with open(os.path.join(BASE_DIR, "processed_v2", "dataset", "dataset_contract.json"), "r", encoding="utf-8") as f:
    dataset_contract = json.load(f)
DATASET_DIR = dataset_contract["dataset_dir"]

with open(dataset_contract["vocab_path"], "r", encoding="utf-8") as f:
    vocab = json.load(f)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: switch to a GPU runtime -- this will be very slow on CPU.")


Mounted at /content/drive
Using device: cuda


## Load the dataset artifacts 

In [3]:
def load_split(name):
    with open(os.path.join(DATASET_DIR, f"{name}_items.pkl"), "rb") as f:
        return pickle.load(f)

splits = {name: load_split(name) for name in ["train", "val", "test"]}
embeddings = np.load(dataset_contract["embeddings_path"]).astype("float32")
node_feat_matrix = np.load(os.path.join(DATASET_DIR, "node_feat_matrix.npy"))

sentiment_to_idx = vocab["sentiment_to_idx"]
emotion_to_idx = vocab["emotion_to_idx"]
idx_to_sentiment = {v: k for k, v in sentiment_to_idx.items()}
idx_to_emotion = {v: k for k, v in emotion_to_idx.items()}
N_SENTIMENT = len(sentiment_to_idx)
N_EMOTION = len(emotion_to_idx)

for name, split in splits.items():
    print(f"{name}: {len(split):,} hyperedges")
print(f"embeddings: {embeddings.shape}, node_feat_matrix: {node_feat_matrix.shape}")
print(f"Sentiment classes ({N_SENTIMENT}): {sentiment_to_idx}")
print(f"Emotion classes ({N_EMOTION}): {emotion_to_idx}")


train: 6,440 hyperedges
val: 1,380 hyperedges
test: 1,380 hyperedges
embeddings: (210994, 768), node_feat_matrix: (22465, 6)
Sentiment classes (3): {'Negative': 0, 'Neutral': 1, 'Positive': 2}
Emotion classes (5): {'Anger': 0, 'Joy': 1, 'Neutral': 2, 'Sadness': 3, 'Surprise': 4}


## Collate function + length-bucketed batch generator

In [4]:
def make_collate_fn(embeddings, node_feat_matrix):
    emb_dim = embeddings.shape[1]
    n_node_feats = node_feat_matrix.shape[1]

    def collate(batch):
        B = len(batch)
        L = max(it["length"] for it in batch)
        emb_batch = np.zeros((B, L, emb_dim), dtype="float32")
        node_feat_batch = np.zeros((B, L, n_node_feats), dtype="float32")
        parent_batch = np.full((B, L), -1, dtype="int64")
        delta_t_batch = np.zeros((B, L), dtype="float32")
        sentiment_batch = np.full((B, L), -1, dtype="int64")
        emotion_batch = np.full((B, L), -1, dtype="int64")
        mask = np.zeros((B, L), dtype="bool")

        for i, it in enumerate(batch):
            n = it["length"]
            emb_batch[i, :n] = embeddings[it["emb_idx"]]
            valid_author = it["author_idx"] >= 0
            node_feat_batch[i, :n][valid_author] = node_feat_matrix[it["author_idx"][valid_author]]
            parent_batch[i, :n] = it["parent_idx"]
            delta_t_batch[i, :n] = it["delta_t"]
            sentiment_batch[i, :n] = it["sentiment_idx"]
            emotion_batch[i, :n] = it["emotion_idx"]
            mask[i, :n] = True

        return {
            "embeddings": torch.from_numpy(emb_batch),
            "node_features": torch.from_numpy(node_feat_batch),
            "parent_idx": torch.from_numpy(parent_batch),
            "delta_t": torch.from_numpy(delta_t_batch),
            "sentiment_idx": torch.from_numpy(sentiment_batch),
            "emotion_idx": torch.from_numpy(emotion_batch),
            "mask": torch.from_numpy(mask),
        }
    return collate

collate_fn = make_collate_fn(embeddings, node_feat_matrix)


def make_length_bucketed_batches(items, batch_size, rng):
    lengths = np.array([it["length"] for it in items])
    order = np.argsort(lengths)  # ascending by length -- similar-length threads end up together
    batches = [order[i:i + batch_size].tolist() for i in range(0, len(order), batch_size)]
    rng.shuffle(batches)  # shuffle batch ORDER only, each epoch -- keeps the efficiency win, adds randomness
    return batches


## Tree-recurrent selective-SSM layer

In [5]:
class TreeRecurrentSSMLayer(nn.Module):
    def __init__(self, d_model, state_dim=16):
        super().__init__()
        self.d_model, self.state_dim = d_model, state_dim
        self.A_log = nn.Parameter(torch.randn(d_model, state_dim) * 0.1)
        self.W_delta = nn.Linear(d_model, d_model)
        self.W_B = nn.Linear(d_model, state_dim)
        self.W_C = nn.Linear(d_model, state_dim)
        self.D = nn.Parameter(torch.randn(d_model) * 0.1)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, parent_idx, mask):
        # x: (B, L, D), parent_idx: (B, L) with -1 for root, mask: (B, L) bool
        B, L, D = x.shape
        N = self.state_dim
        A = -torch.exp(self.A_log)  # (D, N) -- negative real part for a stable/decaying recurrence

        delta = F.softplus(self.W_delta(x))          # (B, L, D)
        Bt = self.W_B(x)                              # (B, L, N)
        Ct = self.W_C(x)                              # (B, L, N)

        Abar = torch.exp(delta.unsqueeze(-1) * A.view(1, 1, D, N))   # (B, L, D, N)
        Bbar = delta.unsqueeze(-1) * Bt.unsqueeze(2)                  # (B, L, D, N)

        # Built via list-append + a final stack, NEVER via in-place slice assignment on a
        # pre-allocated tensor -- in-place mutation of a tensor that's later read by autograd
        # (the gather below) breaks backward() with a "modified by an inplace operation" error.
        batch_arange = torch.arange(B, device=x.device)
        h_history, y_list = [], []

        for t in range(L):
            valid_t = mask[:, t]                       # (B,)
            p = parent_idx[:, t]                        # (B,)
            has_parent = p >= 0                          # (B,)

            if t == 0:
                h_prev = x.new_zeros(B, D, N)
            else:
                h_stack = torch.stack(h_history, dim=1)              # (B, t, D, N)
                gathered = h_stack[batch_arange, p.clamp(min=0, max=t - 1)]
                h_prev = torch.where(has_parent.view(B, 1, 1), gathered, x.new_zeros(B, D, N))

            h_t = Abar[:, t] * h_prev + Bbar[:, t] * x[:, t].unsqueeze(-1)
            h_t = torch.where(valid_t.view(B, 1, 1), h_t, torch.zeros_like(h_t))
            h_history.append(h_t)

            y_t = (Ct[:, t].unsqueeze(1) * h_t).sum(-1) + self.D * x[:, t]
            y_t = torch.where(valid_t.view(B, 1), y_t, torch.zeros_like(y_t))
            y_list.append(y_t)

        y = torch.stack(y_list, dim=1)  # (B, L, D)
        return self.norm(y + x)  # residual + norm, standard practice for stacking these layers


## HOD-Mamba-Lite model

In [6]:
class HODMambaLite(nn.Module):
    def __init__(self, emb_dim, node_feat_dim, d_model=128, state_dim=16, num_layers=4,
                 n_sentiment=3, n_emotion=5):
        super().__init__()
        self.input_proj = nn.Linear(emb_dim, d_model)
        self.node_feat_proj = nn.Linear(node_feat_dim, d_model)
        self.layers = nn.ModuleList([TreeRecurrentSSMLayer(d_model, state_dim) for _ in range(num_layers)])
        self.root_context = nn.Parameter(torch.zeros(d_model))
        self.sentiment_head = nn.Linear(d_model, n_sentiment)
        self.emotion_head = nn.Linear(d_model, n_emotion)

    def forward(self, batch):
        x = self.input_proj(batch["embeddings"]) + self.node_feat_proj(batch["node_features"])
        parent_idx, mask = batch["parent_idx"], batch["mask"]
        for layer in self.layers:
            x = layer(x, parent_idx, mask)
        # x is now the final per-position representation, AFTER seeing each document.
        # Gather the PARENT's final representation for each position -- this is the
        # forecasting input (context available *before* the reply was written).
        B, L, D = x.shape
        batch_arange = torch.arange(B, device=x.device)
        has_parent = parent_idx >= 0
        p = parent_idx.clamp(min=0)
        context = torch.stack([x[b, p[b]] for b in range(B)], dim=0)  # (B, L, D)
        context = torch.where(has_parent.unsqueeze(-1), context, self.root_context.view(1, 1, D))
        return self.sentiment_head(context), self.emotion_head(context)


## Loss and evaluation helpers

In [7]:
def compute_loss(sent_logits, emo_logits, batch):
    parent_idx, mask = batch["parent_idx"], batch["mask"]
    forecastable = mask & (parent_idx >= 0)  # exclude roots -- no in-thread context to forecast them from

    sent_valid = forecastable & (batch["sentiment_idx"] >= 0)
    emo_valid = forecastable & (batch["emotion_idx"] >= 0)

    loss = 0.0
    if sent_valid.any():
        loss = loss + F.cross_entropy(sent_logits[sent_valid], batch["sentiment_idx"][sent_valid])
    if emo_valid.any():
        loss = loss + F.cross_entropy(emo_logits[emo_valid], batch["emotion_idx"][emo_valid])
    return loss, sent_valid, emo_valid


@torch.no_grad()
def evaluate(model, items, batch_size=32):
    model.eval()
    rng = np.random.default_rng(0)
    batches = make_length_bucketed_batches(items, batch_size, rng)
    all_sent_pred, all_sent_true, all_emo_pred, all_emo_true = [], [], [], []

    for batch_idx in batches:
        batch = collate_fn([items[i] for i in batch_idx])
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        sent_logits, emo_logits = model(batch)
        _, sent_valid, emo_valid = compute_loss(sent_logits, emo_logits, batch)

        all_sent_pred.append(sent_logits[sent_valid].argmax(-1).cpu().numpy())
        all_sent_true.append(batch["sentiment_idx"][sent_valid].cpu().numpy())
        all_emo_pred.append(emo_logits[emo_valid].argmax(-1).cpu().numpy())
        all_emo_true.append(batch["emotion_idx"][emo_valid].cpu().numpy())

    sent_pred, sent_true = np.concatenate(all_sent_pred), np.concatenate(all_sent_true)
    emo_pred, emo_true = np.concatenate(all_emo_pred), np.concatenate(all_emo_true)

    sent_f1 = f1_score(sent_true, sent_pred, average="macro", zero_division=0)
    emo_f1 = f1_score(emo_true, emo_pred, average="macro", zero_division=0)
    return sent_f1, emo_f1, (sent_true, sent_pred), (emo_true, emo_pred)


## Training loop

In [8]:
model = HODMambaLite(
    emb_dim=embeddings.shape[1], node_feat_dim=node_feat_matrix.shape[1],
    n_sentiment=N_SENTIMENT, n_emotion=N_EMOTION,
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

BATCH_SIZE = 32
MAX_EPOCHS = 30
PATIENCE = 5
rng = np.random.default_rng(42)

best_combined_f1, best_state, patience_ctr = -1.0, None, 0

for epoch in range(MAX_EPOCHS):
    t0 = time.time()
    model.train()
    batches = make_length_bucketed_batches(splits["train"], BATCH_SIZE, rng)
    total_loss = 0.0

    for batch_idx in batches:
        batch = collate_fn([splits["train"][i] for i in batch_idx])
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        sent_logits, emo_logits = model(batch)
        loss, _, _ = compute_loss(sent_logits, emo_logits, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    val_sent_f1, val_emo_f1, _, _ = evaluate(model, splits["val"], BATCH_SIZE)
    combined_f1 = (val_sent_f1 + val_emo_f1) / 2
    epoch_time = time.time() - t0

    print(f"Epoch {epoch+1}/{MAX_EPOCHS} ({epoch_time:.1f}s): train_loss={total_loss/len(batches):.4f}, "
          f"val_sentiment_f1={val_sent_f1:.3f}, val_emotion_f1={val_emo_f1:.3f}, combined={combined_f1:.3f}")

    if combined_f1 > best_combined_f1:
        best_combined_f1, patience_ctr = combined_f1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no improvement for {PATIENCE} epochs).")
            break

model.load_state_dict(best_state)
print(f"\nBest validation combined macro-F1: {best_combined_f1:.3f}")


Epoch 1/30 (38.0s): train_loss=2.4219, val_sentiment_f1=0.372, val_emotion_f1=0.215, combined=0.293
Epoch 2/30 (36.9s): train_loss=2.2822, val_sentiment_f1=0.406, val_emotion_f1=0.301, combined=0.353
Epoch 3/30 (36.9s): train_loss=2.2480, val_sentiment_f1=0.468, val_emotion_f1=0.308, combined=0.388
Epoch 4/30 (37.0s): train_loss=2.2251, val_sentiment_f1=0.444, val_emotion_f1=0.314, combined=0.379
Epoch 5/30 (36.6s): train_loss=2.2040, val_sentiment_f1=0.430, val_emotion_f1=0.303, combined=0.367
Epoch 6/30 (36.0s): train_loss=2.1961, val_sentiment_f1=0.427, val_emotion_f1=0.283, combined=0.355
Epoch 7/30 (36.7s): train_loss=2.1731, val_sentiment_f1=0.445, val_emotion_f1=0.301, combined=0.373
Epoch 8/30 (37.7s): train_loss=2.1599, val_sentiment_f1=0.450, val_emotion_f1=0.309, combined=0.379
Early stopping at epoch 8 (no improvement for 5 epochs).

Best validation combined macro-F1: 0.388


## Test-set evaluation

In [9]:
test_sent_f1, test_emo_f1, (sent_true, sent_pred), (emo_true, emo_pred) = evaluate(model, splits["test"], BATCH_SIZE)

print(f"Test sentiment macro-F1: {test_sent_f1:.3f}")
print(classification_report(sent_true, sent_pred, target_names=[idx_to_sentiment[i] for i in range(N_SENTIMENT)], zero_division=0))

print(f"\nTest emotion macro-F1: {test_emo_f1:.3f}")
print(classification_report(emo_true, emo_pred, target_names=[idx_to_emotion[i] for i in range(N_EMOTION)], zero_division=0))


Test sentiment macro-F1: 0.479
              precision    recall  f1-score   support

    Negative       0.56      0.69      0.62     11150
     Neutral       0.51      0.47      0.49     10236
    Positive       0.41      0.27      0.33      4922

    accuracy                           0.53     26308
   macro avg       0.49      0.48      0.48     26308
weighted avg       0.51      0.53      0.51     26308


Test emotion macro-F1: 0.312
              precision    recall  f1-score   support

       Anger       0.46      0.63      0.53      6622
         Joy       0.35      0.46      0.40      5302
     Neutral       0.41      0.43      0.42      7866
     Sadness       0.36      0.15      0.21      4776
    Surprise       0.00      0.00      0.00      1742

    accuracy                           0.41     26308
   macro avg       0.32      0.33      0.31     26308
weighted avg       0.37      0.41      0.38     26308



## Save the model and results

In [10]:
MODELS_DIR = os.path.join(BASE_DIR, "processed_v2", "models")
os.makedirs(MODELS_DIR, exist_ok=True)
model_path = os.path.join(MODELS_DIR, "hod_mamba_lite.pt")
torch.save(model.state_dict(), model_path)

results = {
    "model": "HOD-Mamba-Lite",
    "hyperparameters": {"d_model": 128, "state_dim": 16, "num_layers": 4, "batch_size": BATCH_SIZE,
                         "lr": 1e-3, "max_epochs": MAX_EPOCHS, "patience": PATIENCE},
    "best_val_combined_macro_f1": float(best_combined_f1),
    "test_sentiment_macro_f1": float(test_sent_f1),
    "test_emotion_macro_f1": float(test_emo_f1),
    "model_path": model_path,
}
with open(os.path.join(MODELS_DIR, "hod_mamba_lite_results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Saved:", model_path)
print(json.dumps(results, indent=2, ensure_ascii=False))


Saved: /content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/models/hod_mamba_lite.pt
{
  "model": "HOD-Mamba-Lite",
  "hyperparameters": {
    "d_model": 128,
    "state_dim": 16,
    "num_layers": 4,
    "batch_size": 32,
    "lr": 0.001,
    "max_epochs": 30,
    "patience": 5
  },
  "best_val_combined_macro_f1": 0.3881581924348856,
  "test_sentiment_macro_f1": 0.4786653565055463,
  "test_emotion_macro_f1": 0.31189183110798313,
  "model_path": "/content/drive/MyDrive/PhD/reddit_opinion_research/processed_v2/models/hod_mamba_lite.pt"
}


## The results


In [11]:
from sklearn.metrics import confusion_matrix
import pandas as pd

emotion_names = [idx_to_emotion[i] for i in range(N_EMOTION)]
cm = pd.DataFrame(confusion_matrix(emo_true, emo_pred, labels=list(range(N_EMOTION))),
                   index=emotion_names, columns=emotion_names)
print(cm)
print("\nSurprise's true examples, predicted as:")
print(cm.loc["Surprise"])

          Anger   Joy  Neutral  Sadness  Surprise
Anger      4153   671     1398      400         0
Joy         903  2444     1627      328         0
Neutral    2120  1909     3420      417         0
Sadness    1368  1351     1356      701         0
Surprise    503   527      631       81         0

Surprise's true examples, predicted as:
Anger       503
Joy         527
Neutral     631
Sadness      81
Surprise      0
Name: Surprise, dtype: int64
